# Agent System Explorer

This notebook helps you explore and debug the agent system interactively.

**Prerequisites:**
- Run `uv sync` to install dependencies
- Copy `.env.example` to `.env` and fill in API keys

In [ ]:
# Load environment variables
import sys
from pathlib import Path

# Ensure project root is on the Python path
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

## 1. Explore the Registry

View all registered sub-agents and their configurations.

In [ ]:
from agents.registry import AgentRegistry, SubAgentConfig
from subagents.general import create_general_subagent

# Build registry
registry = AgentRegistry()
general = create_general_subagent()
registry.register(SubAgentConfig(
    name=general["name"],
    description=general["description"],
    system_prompt=general["system_prompt"],
    tools=general["tools"],
))

# Display
print("Registered sub-agents:")
for a in registry.list_agents():
    tools_names = [getattr(t, 'name', str(t)) for t in a.tools]
    print(f"  📌 {a.name}")
    print(f"     Description: {a.description}")
    print(f"     Tools: {', '.join(tools_names) or '(default)'}")
    print()

print("Capabilities summary:")
print(registry.capabilities_summary())

## 2. Test the Router

Test rule-based and LLM-based routing.

In [ ]:
from agents.config import AgentConfig
from agents.router import IntentRouter, RoutingRule

config = AgentConfig(routing_strategy="rule_first")
router = IntentRouter(config=config, registry=registry, llm_model=None)

# Add a test rule
router.add_rule(RoutingRule(
    patterns=["hello|hi|你好|greet"],
    agent_name="general",
    description="Greeting patterns",
))

# Test routing
test_queries = [
    "你好，帮我查一下今天的天气",
    "Hello! Can you help me?",
    "Write a Python sorting algorithm",
    "随便聊聊",
]

for query in test_queries:
    match = router.route(query)
    print(f"Query: {query!r}")
    print(f"  -> Agent: {match.agent_name} (source: {match.source}, confidence: {match.confidence})")
    print()

## 3. Create and Run the Agent

Build the full orchestrator and run a test query.

In [ ]:
from agents.orchestrator import create_orchestrator

# Create the orchestrator (requires API keys in .env)
agent = create_orchestrator(
    config=config,
    registry=registry,
)
print("✅ Orchestrator agent created successfully!")
print(f"   Type: {type(agent).__name__}")

In [ ]:
# Run a test query (streaming)
test_query = "你好，介绍一下你自己能做什么"

print(f"\n📝 User: {test_query}\n")
print("🤖 Assistant:")

for event in agent.stream(
    {"messages": [{"role": "user", "content": test_query}]},
    stream_mode="updates",
):
    # Print relevant events
    for key, value in event.items():
        if "messages" in value:
            for msg in value["messages"]:
                if hasattr(msg, "content") and msg.content:
                    role = getattr(msg, "type", "unknown")
                    if role == "ai":
                        print(msg.content, end="", flush=True)
print()